In [ ]:
import json
from datetime import datetime, date, time, timedelta

# ---------------------------------------------------------------------------
# 1. BUILD THE CALENDAR OF OPEN TICKS (from earlier step)
# ---------------------------------------------------------------------------

BREAKS = [
    (time(9, 0), time(9, 15)),
    (time(13, 0), time(13, 30)),
    (time(14, 0), time(14, 15)),
    (time(17, 0), time(17, 15)),
    (time(19, 0), time(19, 30)),
    (time(23, 0), time(23, 15)),
]
HOLIDAYS = {date(2026, 7, 15)}


def is_open(dt):
    
    t = dt.time()

    if time(1, 0) <= t < time(7, 0):
        return False  # dead zone between shifts

    shift_day = dt.date() - timedelta(days=1) if t < time(1, 0) else dt.date()

    if shift_day.weekday() >= 5 or shift_day in HOLIDAYS:
        return False

    for start, end in BREAKS:
        if start <= t < end:
            return False

    return True


def build_ticks(start, end, step=timedelta(hours=0.01)):
    ticks = []
    t = start
    while t < end:
        if is_open(t):
            ticks.append(t)
        t += step
    return ticks


# ---------------------------------------------------------------------------
# 2. LOAD DATA
# ---------------------------------------------------------------------------

# Swap these for json.load(open("jobs.json")) / json.load(open("work_centers.json"))
# when you're reading from actual files.
with open("jobs.json") as f:
    jobs = json.load(f)

with open("work_centers.json") as f:
    work_centers = json.load(f)

DEPT_MAP = {
    1: work_centers["dept_1_work_centers"],
    2: work_centers["dept_2_work_centers"],
}


# ---------------------------------------------------------------------------
# 3. CAPABILITY CHECK
# ---------------------------------------------------------------------------

def qualifies(wc, department, material):
    if department == 1:
        key = "Steel" if material == "steel" else "Aluminum"
    elif department == 2:
        key = "Weld_Steel" if material == "steel" else "Weld_Aluminum"
    else:
        return False  # dept 3 / 4 not used for this job data
    return wc.get(key, False)


# ---------------------------------------------------------------------------
# 4. SCHEDULER
# ---------------------------------------------------------------------------

def schedule_jobs(jobs, dept_map, ticks):
    # Earliest due date first
    jobs_sorted = sorted(jobs, key=lambda j: j["due_date"])

    # Each work center's cursor: earliest tick-list index it's free at
    wc_cursor = {wc["Number"]: 0 for dept in dept_map.values() for wc in dept}

    results = []

    for job in jobs_sorted:
        op_ready_index = 0
        op_results = []
        unscheduled = False

        for op in sorted(job["operations"], key=lambda o: o["op_seq"]):
            department = op["department"]
            material = job["stock_material"]

            candidates = [wc for wc in dept_map[department]
                          if qualifies(wc, department, material)]

            if not candidates:
                op_results.append({
                    "op_seq": op["op_seq"],
                    "status": f"no work center qualifies (dept {department}, {material})",
                })
                unscheduled = True
                break

            duration_ticks = round((op["setup_time"] + op["run_time"]) * 100)

            # Pick whichever qualifying work center is free soonest
            best_wc, best_start = None, None
            for wc in candidates:
                wc_num = wc["Number"]
                start_index = max(op_ready_index, wc_cursor[wc_num])
                if best_start is None or start_index < best_start:
                    best_start, best_wc = start_index, wc_num

            end_index = best_start + duration_ticks

            if end_index > len(ticks):
                op_results.append({
                    "op_seq": op["op_seq"],
                    "status": "unscheduled - runs past end of calendar window",
                })
                unscheduled = True
                break

            start_time = ticks[best_start]
            finish_time = ticks[end_index - 1]

            wc_cursor[best_wc] = end_index
            op_ready_index = end_index

            op_results.append({
                "op_seq": op["op_seq"],
                "work_center": best_wc,
                "start": start_time,
                "finish": finish_time,
                "duration_ticks": duration_ticks,
            })

        due = date.fromisoformat(job["due_date"])
        first_start = op_results[0].get("start") if op_results else None
        last_finish = op_results[-1].get("finish") if op_results else None
        late = (last_finish.date() > due) if last_finish else None

        results.append({
            "job_number": job["job_number"],
            "due_date": due,
            "start": first_start,
            "finish": last_finish,
            "late": late,
            "unscheduled": unscheduled,
            "operations": op_results,
        })

    return results


# ---------------------------------------------------------------------------
# 5. OUTPUT
# ---------------------------------------------------------------------------

def write_outputs(results, summary_path="schedule_summary.csv", detail_path="schedule_detail.csv"):
    """
    summary_path: one row per job - what a manager scans first.
    detail_path:  one row per operation - the full backing detail.
    """
    import pandas as pd

    summary_rows = []
    detail_rows = []

    for r in results:
        summary_rows.append({
            "job_number": r["job_number"],
            "start": r["start"].strftime("%Y-%m-%d %H:%M") if r["start"] else None,
            "due_date": r["due_date"],
            "expected_finish": r["finish"].strftime("%Y-%m-%d %H:%M") if r["finish"] else None,
            "status": "LATE" if r["late"] else ("ON TIME" if r["late"] is False else "UNSCHEDULED"),
        })

        for op in r["operations"]:
            row = {"job_number": r["job_number"], "op_seq": op["op_seq"]}
            if "status" in op:
                row["status"] = op["status"]
            else:
                row.update({
                    "work_center": op["work_center"],
                    "start": op["start"].strftime("%Y-%m-%d %H:%M"),
                    "finish": op["finish"].strftime("%Y-%m-%d %H:%M"),
                })
            detail_rows.append(row)

    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
    pd.DataFrame(detail_rows).to_csv(detail_path, index=False)
    print(f"Wrote {summary_path} ({len(summary_rows)} jobs)")
    print(f"Wrote {detail_path} ({len(detail_rows)} operations)")


# ---------------------------------------------------------------------------
# 6. RUN
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    ticks = build_ticks(datetime(2026, 7, 1, 0, 0), datetime(2026, 7, 22, 0, 0))
    results = schedule_jobs(jobs, DEPT_MAP, ticks)
    write_outputs(results)